# Flood vs Permanent Water Segmentation — Training (v3)

3-class semantic segmentation (`0=non-water`, `1=flood`, `2=permanent water`) on Sen1Floods11 with all 6 channels (S1-VV, S1-VH, DEM, Slope, JRC, HAND).

**Datasets to attach in the Kaggle notebook sidebar:**
- `yash10chawla/final-sendata` (the 7-band canonical dataset)
- `yash10chawla/flood-detection-src` (latest version of the .py files)
- `yash10chawla/last-try-key` (only if you'll run inference via GEE — not needed for training)

## 1. Paths and dependencies

In [ ]:
import os, sys

SRC_DIR        = '/kaggle/input/datasets/yash10chawla/flood-detection-src'
DATA_ROOT      = '/kaggle/input/datasets/yash10chawla/final-sendata'
SPLITS_ROOT    = f'{DATA_ROOT}/splits'
CHECKPOINT_DIR = '/kaggle/working/checkpoints-v3'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
sys.path.insert(0, SRC_DIR)

for p in (SRC_DIR, DATA_ROOT, SPLITS_ROOT):
    assert os.path.isdir(p), f'NOT FOUND: {p}'
print('All paths OK.')
print(f'  SRC_DIR        = {SRC_DIR}')
print(f'  DATA_ROOT      = {DATA_ROOT}')
print(f'  SPLITS_ROOT    = {SPLITS_ROOT}')
print(f'  CHECKPOINT_DIR = {CHECKPOINT_DIR}')

In [ ]:
!pip install -q timm albumentations rasterio einops

## 2. Sanity check — dataset audit + class distribution

Before training, verify two things:
1. **All 7 band directories are populated** — the audit log will show MISSING/EMPTY for any dir that's not there.
2. **The 3-class label actually contains class-2 pixels** — if the JRC threshold is wrong for your data, `perm` count will be tiny and val_IoU(perm) will stay at 0 forever.

If `perm` pixel fraction is **<0.5%** of foreground, lower `JRC_PERMANENT_THRESHOLD` in `dataset.py` (try 3 or 4 instead of 5).

In [ ]:
from dataset import build_datasets
import torch
from collections import Counter

datasets = build_datasets(
    data_root   = DATA_ROOT,
    splits_root = SPLITS_ROOT,
    patch_size  = 512,
)

# Sample 20 train chips, count pixel classes — confirms 3-class labels work
ds = datasets['train']
totals = Counter()
n_check = min(20, len(ds))
for i in range(n_check):
    label = ds[i]['label']
    for v, c in zip(*torch.unique(label, return_counts=True)):
        totals[int(v)] += int(c)

names = {-1: 'ignore', 0: 'non-water', 1: 'flood', 2: 'permanent'}
total = sum(totals.values())
fg_total = totals.get(1, 0) + totals.get(2, 0)
print(f'Class distribution across {n_check} train chips ({total:,} pixels):')
for cls in (-1, 0, 1, 2):
    n = totals.get(cls, 0)
    pct = 100 * n / max(total, 1)
    print(f'  class {cls:>2} ({names[cls]:>10s}): {n:>10,d} pixels  ({pct:5.2f}%)')

if fg_total > 0:
    perm_frac = totals.get(2, 0) / fg_total
    print(f'\nPermanent share of foreground (flood + permanent): {perm_frac*100:.2f}%')
    if perm_frac < 0.005:
        print('  ⚠️  WARNING: very few permanent-water pixels.')
        print('     Lower JRC_PERMANENT_THRESHOLD in dataset.py (try 3 or 4) and re-upload the src dataset.')
    else:
        print('  ✅ Looks fine — permanent class has enough signal to learn.')
else:
    print('\n❌ No foreground pixels at all in sampled chips — something is very wrong with labels.')

## 3. One-batch shape sanity (cheap forward+backward pass)

Catches model/loss/dtype bugs in <30 seconds before launching the long run.

In [ ]:
from torch.utils.data import DataLoader
from model  import build_model
from losses import get_mtl_loss
from metrics import MetricTracker

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

loader = DataLoader(datasets['train'], batch_size=2, shuffle=True, num_workers=0)
batch  = next(iter(loader))
print(f"image: {batch['image'].shape}  label: {batch['label'].shape}  mndwi: {batch['mndwi'].shape}")

model     = build_model(in_channels=6, pretrained=True).to(device)
criterion = get_mtl_loss('tversky')

logits, mndwi_pred = model(batch['image'].to(device))
loss = criterion(logits, mndwi_pred, batch['label'].to(device), batch['mndwi'].to(device))
loss.backward()

tracker = MetricTracker()
tracker.update(torch.softmax(logits, dim=1), batch['label'].to(device))
metrics = tracker.compute()

print(f'\nlogits: {logits.shape}  loss: {loss.item():.4f}')
print('metrics keys:', list(metrics.keys()))
assert logits.shape[1] == 3, 'model is not 3-class — re-attach latest flood-detection-src dataset version'
print('\n✅ Forward+backward pass OK. Safe to train.')

del model, logits, mndwi_pred, loss, tracker
torch.cuda.empty_cache()

## 4. Train (4-loss model soup)

Trains 4 separate models (one per loss: dice / jaccard / tversky / focal_tversky), then builds a greedy weight-averaged soup. Total time on a P100: ~6–10 hours for 400 epochs, but early stopping (`patience_stop=20`) usually cuts each run short.

If you only have time for one run: pass `'loss_names': ['tversky']` in CFG.

In [ ]:
from train import main as train_main, DEFAULT_CONFIG

CFG = {
    **DEFAULT_CONFIG,
    'data_root':       DATA_ROOT,
    'splits_root':     SPLITS_ROOT,
    'checkpoint_dir':  CHECKPOINT_DIR,
    'batch_size':      12,
    'num_workers':     4,
    'max_epochs':      400,
    'patience_stop':   20,
    'patience_reduce': 8,
    'lr':              1e-3,
    # 'loss_names':    ['tversky'],   # uncomment to skip the model soup and only train one loss
}

soup_model, test_metrics = train_main(CFG)
print('\nFinal test metrics:')
for k, v in test_metrics.items():
    print(f'  {k:18s} = {v:.4f}')

## 5. Inference demo — render red-flood / blue-permanent map on a test chip

Picks one chip from the test split, builds a 6-band stack from the disk files, runs the trained model, and saves the visualization.

In [ ]:
import numpy as np, rasterio
from inference import (FloodPredictor, sliding_window_predict, postprocess,
                       visualize_flood_map, load_and_normalize)

# Pick any base ID from the test split
test_chip_id = datasets['test'].samples[0]
print(f'Demo chip: {test_chip_id}')

# Build a 6-band stack on disk so inference.load_and_normalize can read it
DEMO_TIF = f'/kaggle/working/{test_chip_id}_6band.tif'
BAND_FILES = {
    'S1Hand':       2,   # 2 bands (VV, VH) → channels 1,2
    'DEM':          1,   # → channel 3
    'Slope':        1,   # → channel 4
    'JRCWaterHand': 1,   # → channel 5
    'HAND':         1,   # → channel 6
}

stacked = []
ref_profile = None
for subdir, n_bands in BAND_FILES.items():
    suffix = subdir if subdir != 'S1Hand' else 'S1Hand'
    p = f'{DATA_ROOT}/HandLabeled/{subdir}/{test_chip_id}_{suffix}.tif'
    with rasterio.open(p) as src:
        if ref_profile is None:
            ref_profile = src.profile.copy()
        for b in range(1, n_bands + 1):
            stacked.append(src.read(b).astype(np.float32))

ref_profile.update({'count': 6, 'dtype': 'float32'})
with rasterio.open(DEMO_TIF, 'w', **ref_profile) as dst:
    for i, arr in enumerate(stacked, 1):
        dst.write(arr, i)
print(f'Built 6-band stack: {DEMO_TIF}')

predictor = FloodPredictor(f'{CHECKPOINT_DIR}/model_soup.pth')
results = predictor.predict_from_geotiff(DEMO_TIF, '/kaggle/working/inference_outputs')

from IPython.display import Image
Image(results['png'])

## 6. Snapshot checkpoints to a Kaggle dataset (optional)

After training, the soup + 4 individual checkpoints are in `/kaggle/working/checkpoints-v3/`. Manually create a new Kaggle dataset from this folder so you can attach it in inference notebooks without retraining.

In [ ]:
!ls -lh /kaggle/working/checkpoints-v3/